# Clase 6 — Criptografía Asimétrica: RSA

> **Curso:** Criptografía Aplicada  
> **Objetivo:** Comprender el diseño matemático de RSA, su implementación práctica con Python, los esquemas de relleno seguros (OAEP, PSS) y los principales ataques clásicos que motivan el uso correcto del algoritmo.

---

## Tabla de contenidos

1. [Introducción a la criptografía asimétrica](#1)  
2. [Fundamentos matemáticos de RSA](#2)  
   2.1 [Función de Euler y de Carmichael](#2-1)  
   2.2 [Teorema de Euler y pequeño Fermat](#2-2)  
   2.3 [Test de primalidad de Miller-Rabin](#2-3)  
3. [Generación de claves RSA desde cero](#3)  
   3.1 [Algoritmo paso a paso](#3-1)  
   3.2 [Implementación educativa en Python puro](#3-2)  
   3.3 [Formato PEM y estructura ASN.1](#3-3)  
4. [Cifrado y descifrado RSA](#4)  
   4.1 [RSA textbook (sin relleno) — solo educativo](#4-1)  
   4.2 [PKCS#1 v1.5 — cifrado](#4-2)  
   4.3 [OAEP (Optimal Asymmetric Encryption Padding)](#4-3)  
5. [Firma digital con RSA](#5)  
   5.1 [PKCS#1 v1.5 — firma](#5-1)  
   5.2 [PSS (Probabilistic Signature Scheme)](#5-2)  
   5.3 [Comparativa PKCS#1 v1.5 vs PSS](#5-3)  
6. [RSA con la biblioteca `cryptography`](#6)  
   6.1 [Generación, serialización y carga de claves](#6-1)  
   6.2 [Cifrado OAEP completo](#6-2)  
   6.3 [Firma y verificación PSS](#6-3)  
   6.4 [Cifrado híbrido RSA + AES-GCM](#6-4)  
7. [Ataques clásicos a RSA](#7)  
   7.1 [Ataque de exponente público pequeño (e=3)](#7-1)  
   7.2 [Broadcast attack (Håstad)](#7-2)  
   7.3 [Factorización: Pollard's rho](#7-3)  
   7.4 [Módulos compartidos (Common Modulus)](#7-4)  
   7.5 [Ataque de Wiener (exponente privado pequeño)](#7-5)  
   7.6 [Bleichenbacher (PKCS#1 v1.5 oracle)](#7-6)  
8. [Buenas prácticas y selección de parámetros](#8)  
9. [Referencias](#9)

---

## Importaciones globales

In [ ]:
import math, random, os, hashlib, struct, time, secrets, base64, textwrap
from functools import reduce

# Biblioteca cryptography (estándar de producción)
from cryptography.hazmat.primitives.asymmetric import rsa as _rsa_lib, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric.rsa import generate_private_key
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidSignature

print("Importaciones correctas ✓")

---
<a id='1'></a>
## 1. Introducción a la criptografía asimétrica

La **criptografía de clave pública** (o *asimétrica*) fue propuesta por Diffie y Hellman en 1976 y resolvió el problema fundamental de la distribución de claves simétricas.

### Idea central

En lugar de compartir un único secreto, cada entidad posee un **par de claves**:

| Clave | Visibilidad | Uso |
|---|---|---|
| Clave pública `pk` | Todo el mundo puede conocerla | Cifrar mensajes, verificar firmas |
| Clave privada `sk` | Solo la conoce el propietario | Descifrar mensajes, firmar |

La seguridad descansa en la dificultad computacional de ciertos problemas matemáticos:

| Algoritmo | Problema duro |
|---|---|
| RSA | Factorización de enteros grandes |
| Diffie-Hellman / ElGamal | Logaritmo discreto en $\mathbb{Z}_p^*$ |
| ECDH / ECDSA | Logaritmo discreto en curvas elípticas |
| NTRU / Kyber | Retícula (lattice) — resistente a cuánticos |

### ¿Por qué usar cifrado híbrido?

RSA es lento para mensajes largos. La práctica estándar es:

```
1. Generar clave de sesión aleatoria K (128/256 bits)
2. Cifrar K con RSA-OAEP → C_K  (se envía junto al mensaje)
3. Cifrar mensaje con AES-GCM usando K → C_M
4. Transmitir (C_K, C_M)
```

> **TLS 1.3** ya no usa RSA para intercambio de clave (usa ECDHE); RSA solo se usa para autenticación/firma.

---
<a id='2'></a>
## 2. Fundamentos matemáticos de RSA

RSA se apoya en la **aritmética modular** y la **teoría de números**. Repasamos los conceptos imprescindibles.

<a id='2-1'></a>
### 2.1 Función de Euler $\phi$ y función de Carmichael $\lambda$

Dado $n = p \cdot q$ con $p, q$ primos:

$$\phi(n) = (p-1)(q-1)$$

$$\lambda(n) = \mathrm{lcm}(p-1,\, q-1)$$

El estándar moderno usa $\lambda(n)$ porque produce exponentes privados más pequeños (y equivalentes en seguridad).

In [ ]:
def euler_totient(p: int, q: int) -> int:
    """Función de Euler para n = p*q."""
    return (p - 1) * (q - 1)

def carmichael_lambda(p: int, q: int) -> int:
    """Función de Carmichael para n = p*q."""
    return math.lcm(p - 1, q - 1)

p, q = 61, 53
n = p * q
phi = euler_totient(p, q)
lam = carmichael_lambda(p, q)

print(f"n = p·q = {p} · {q} = {n}")
print(f"φ(n)   = (p-1)(q-1)      = {phi}")
print(f"λ(n)   = lcm(p-1, q-1)   = {lam}")
print(f"φ(n)/λ(n) = {phi//lam}  (λ siempre divide a φ)")

<a id='2-2'></a>
### 2.2 Teorema de Euler y pequeño Fermat

**Teorema de Euler:** Si $\gcd(a, n) = 1$, entonces $a^{\phi(n)} \equiv 1 \pmod{n}$.

**Corolario (pequeño Fermat):** Si $p$ es primo y $\gcd(a, p) = 1$, entonces $a^{p-1} \equiv 1 \pmod{p}$.

Esto garantiza que $m^{ed} \equiv m \pmod{n}$ cuando $ed \equiv 1 \pmod{\lambda(n)}$.

In [ ]:
# Verificación empírica del teorema de Euler
p2, q2 = 97, 89
n2 = p2 * q2
lam2 = carmichael_lambda(p2, q2)

a = 1234   # mensaje de prueba
assert math.gcd(a, n2) == 1, "a debe ser coprimo con n"

result = pow(a, lam2, n2)
print(f"a^λ(n) mod n = {a}^{lam2} mod {n2} = {result}  (esperado: 1)")

# Construcción RSA: e·d ≡ 1 (mod λ(n))
e = 65537
d = pow(e, -1, lam2)   # inverso modular — Python 3.8+
print(f"\ne = {e}")
print(f"d = {d}")
print(f"e·d mod λ(n) = {(e*d) % lam2}  (esperado: 1)")

# Cifrado y descifrado de prueba
m = 42
c = pow(m, e, n2)
m_dec = pow(c, d, n2)
print(f"\nMensaje original : {m}")
print(f"Cifrado          : {c}")
print(f"Descifrado       : {m_dec}  ({'✓' if m_dec == m else '✗'})")

<a id='2-3'></a>
### 2.3 Test de primalidad de Miller-Rabin

Para generar primos RSA necesitamos un test eficiente. El test de **Miller-Rabin** es probabilístico: con $k$ testigos aleatorios, la probabilidad de un falso positivo es $\leq 4^{-k}$.

> Para claves RSA de 2048 bits se recomienda $k \geq 40$ rondas.

In [ ]:
def miller_rabin(n: int, k: int = 40) -> bool:
    """Test de Miller-Rabin. Devuelve True si n es probablemente primo."""
    if n < 2:
        return False
    if n == 2 or n == 3:
        return True
    if n % 2 == 0:
        return False

    # Escribir n-1 = 2^r · d
    r, d = 0, n - 1
    while d % 2 == 0:
        r += 1
        d //= 2

    for _ in range(k):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False   # compuesto seguro
    return True            # probablemente primo


def generate_prime(bits: int) -> int:
    """Genera un primo aleatorio de `bits` bits usando Miller-Rabin."""
    while True:
        candidate = random.getrandbits(bits)
        candidate |= (1 << bits - 1) | 1   # forzar bit alto y bit bajo (impar)
        if miller_rabin(candidate):
            return candidate


# Prueba
import time
for bits in [128, 256, 512]:
    t0 = time.perf_counter()
    p = generate_prime(bits)
    elapsed = time.perf_counter() - t0
    print(f"{bits}-bit primo generado en {elapsed*1000:.1f} ms: {hex(p)[:20]}...")

---
<a id='3'></a>
## 3. Generación de claves RSA desde cero

<a id='3-1'></a>
### 3.1 Algoritmo paso a paso

| Paso | Operación |
|---|---|
| 1 | Elegir dos primos grandes $p \neq q$ de igual tamaño en bits |
| 2 | Calcular $n = p \cdot q$ (módulo público) |
| 3 | Calcular $\lambda(n) = \mathrm{lcm}(p-1, q-1)$ |
| 4 | Elegir $e$ tal que $1 < e < \lambda(n)$ y $\gcd(e, \lambda(n)) = 1$ |
| 5 | Calcular $d = e^{-1} \bmod \lambda(n)$ |
| **Clave pública** | $(n, e)$ |
| **Clave privada** | $(n, d)$ — también se guardan $p, q, d_p, d_q, q_{inv}$ para CRT |

**Parámetros CRT** (aceleran el descifrado ≈ 4×):

$$d_p = d \bmod (p-1), \quad d_q = d \bmod (q-1), \quad q_{inv} = q^{-1} \bmod p$$

<a id='3-2'></a>
### 3.2 Implementación educativa en Python puro

> ⚠️ Esta implementación es **solo educativa**. No está protegida contra ataques de canal lateral (timing, cache). Para producción usar siempre `cryptography` o `openssl`.

In [ ]:
class RSAKey:
    """Par de claves RSA educativo (NO usar en producción)."""

    def __init__(self, bits: int = 512):
        assert bits % 2 == 0 and bits >= 256, "bits debe ser par y ≥ 256"
        half = bits // 2

        # 1. Generar primos p y q de half bits
        print(f"Generando primos de {half} bits...")
        p = generate_prime(half)
        q = generate_prime(half)
        while q == p:
            q = generate_prime(half)

        # Convención: p > q
        if p < q:
            p, q = q, p

        # 2. Módulo
        n = p * q

        # 3. λ(n)
        lam = math.lcm(p - 1, q - 1)

        # 4. Exponente público e = 65537 (primer de Fermat F4)
        e = 65537
        assert math.gcd(e, lam) == 1, "e no es coprimo con λ(n)"

        # 5. Exponente privado
        d = pow(e, -1, lam)

        # Parámetros CRT
        dp  = d % (p - 1)
        dq  = d % (q - 1)
        qinv = pow(q, -1, p)

        # Almacenar
        self.n, self.e, self.d = n, e, d
        self.p, self.q = p, q
        self.dp, self.dq, self.qinv = dp, dq, qinv
        self.bits = bits

    # ── Cifrado/descifrado textbook (sin padding) ─────────────────────────
    def encrypt_raw(self, m: int) -> int:
        assert 0 < m < self.n, "mensaje fuera de rango"
        return pow(m, self.e, self.n)

    def decrypt_raw_crt(self, c: int) -> int:
        """Descifrado usando CRT (más rápido)."""
        # m1 = c^dp mod p,  m2 = c^dq mod q
        m1 = pow(c, self.dp, self.p)
        m2 = pow(c, self.dq, self.q)
        # Recombinar con CRT
        h  = (self.qinv * (m1 - m2)) % self.p
        m  = m2 + h * self.q
        return m

    def decrypt_raw(self, c: int) -> int:
        return pow(c, self.d, self.n)

    def key_info(self):
        print(f"Tamaño clave  : {self.bits} bits")
        print(f"n ({self.n.bit_length()} bits): {hex(self.n)[:30]}...")
        print(f"e             : {self.e}")
        print(f"d ({self.d.bit_length()} bits): {hex(self.d)[:30]}...")
        print(f"p ({self.p.bit_length()} bits): {hex(self.p)[:20]}...")
        print(f"q ({self.q.bit_length()} bits): {hex(self.q)[:20]}...")


# Generar clave de 512 bits (educativa, pequeña)
t0 = time.perf_counter()
rsa_key = RSAKey(512)
print(f"Clave generada en {(time.perf_counter()-t0)*1000:.1f} ms\n")
rsa_key.key_info()

In [ ]:
# Verificar cifrado/descifrado textbook
m = 123456789
c = rsa_key.encrypt_raw(m)
m_std = rsa_key.decrypt_raw(c)
m_crt = rsa_key.decrypt_raw_crt(c)

print(f"Mensaje original : {m}")
print(f"Cifrado (hex)    : {hex(c)[:40]}...")
print(f"Descifrado std   : {m_std}  {'✓' if m_std == m else '✗'}")
print(f"Descifrado CRT   : {m_crt}  {'✓' if m_crt == m else '✗'}")

# Benchmark CRT vs naive
import timeit
N = 200
t_std = timeit.timeit(lambda: rsa_key.decrypt_raw(c), number=N)
t_crt = timeit.timeit(lambda: rsa_key.decrypt_raw_crt(c), number=N)
print(f"\nBenchmark ({N} iters):")
print(f"  Descifrado naive : {t_std*1000/N:.3f} ms/op")
print(f"  Descifrado CRT   : {t_crt*1000/N:.3f} ms/op")
print(f"  Aceleración CRT  : {t_std/t_crt:.2f}×")

<a id='3-3'></a>
### 3.3 Formato PEM y estructura ASN.1

Las claves RSA se codifican en **DER** (Distinguished Encoding Rules, binario ASN.1) y luego se envuelven en **PEM** (base64 + cabeceras).

In [ ]:
# Generar clave RSA 2048 bits con cryptography y serializarla
private_key = generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
public_key = private_key.public_key()

# Serializar clave privada a PEM (sin passphrase)
pem_private = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=serialization.NoEncryption(),
)

# Serializar clave pública a PEM
pem_public = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)

print("=== CLAVE PRIVADA (primeras líneas) ===")
lines = pem_private.decode().strip().split("\n")
for l in lines[:6]:
    print(l)
print("  ...")
print(lines[-1])

print("\n=== CLAVE PÚBLICA ===")
print(pem_public.decode().strip())

In [ ]:
# Inspeccionar los números internos de la clave
priv_numbers = private_key.private_numbers()
pub_numbers  = public_key.public_numbers()

print(f"Módulo n ({pub_numbers.n.bit_length()} bits): {hex(pub_numbers.n)[:40]}...")
print(f"Exponente público e : {pub_numbers.e}")
print(f"Exponente privado d ({priv_numbers.d.bit_length()} bits): {hex(priv_numbers.d)[:40]}...")
print(f"p ({priv_numbers.p.bit_length()} bits): {hex(priv_numbers.p)[:30]}...")
print(f"q ({priv_numbers.q.bit_length()} bits): {hex(priv_numbers.q)[:30]}...")
print(f"dp = d mod (p-1)    : {hex(priv_numbers.dmp1)[:30]}...")
print(f"dq = d mod (q-1)    : {hex(priv_numbers.dmq1)[:30]}...")
print(f"qInv = q^-1 mod p   : {hex(priv_numbers.iqmp)[:30]}...")

---
<a id='4'></a>
## 4. Cifrado y descifrado RSA

<a id='4-1'></a>
### 4.1 RSA textbook (sin relleno) — solo educativo

El cifrado *textbook* es **deterministico** y tiene múltiples vulnerabilidades:

- Mensajes iguales producen cifrados iguales (**no IND-CPA**).
- $m = 0, 1, n-1$ son puntos fijos.
- Para $e = 3$ y mensajes pequeños, $c = m^3$ puede resolverse con una raíz cúbica entera.
- Es **maleable**: dado $c = m^e \bmod n$, el atacante puede construir $c' = c \cdot 2^e \bmod n$ que descifra a $2m$.

In [ ]:
# Demostración: RSA textbook es determinístico (no IND-CPA)
msg = 42
c1 = rsa_key.encrypt_raw(msg)
c2 = rsa_key.encrypt_raw(msg)
print(f"Cifrado 1: {c1}")
print(f"Cifrado 2: {c2}")
print(f"¿Iguales? {c1 == c2}  → Vulnerabilidad IND-CPA")

# Demostración: maleabilidad
factor = 3
c_maleable = (c1 * pow(factor, rsa_key.e, rsa_key.n)) % rsa_key.n
m_maleable  = rsa_key.decrypt_raw(c_maleable)
print(f"\nMensaje original: {msg}")
print(f"Descifrado maleable: {m_maleable}  (= {factor} × {msg} = {factor*msg}) {'✓' if m_maleable == factor*msg else '✗'}") 

<a id='4-2'></a>
### 4.2 PKCS#1 v1.5 — cifrado

Definido en [RFC 8017](https://www.rfc-editor.org/rfc/rfc8017). Añade un relleno pseudoaleatorio que rompe el determinismo.

**Estructura del bloque cifrado** (para $n$ de $k$ octetos):

```
0x00 || 0x02 || PS (≥ 8 bytes no nulos aleatorios) || 0x00 || M
```

- **PS**: al menos 8 bytes no nulos aleatorios.
- Longitud máxima de mensaje: $k - 11$ bytes.

> ⚠️ PKCS#1 v1.5 es vulnerable al ataque de **Bleichenbacher** (1998) si el servidor revela información sobre el formato. Ver sección 7.6.

In [ ]:
# PKCS#1 v1.5 cifrado con cryptography
message = b"Mensaje secreto RSA PKCS1v15"

# Cifrar con clave pública
ciphertext_pkcs1 = public_key.encrypt(
    message,
    asym_padding.PKCS1v15()
)

print(f"Mensaje        ({len(message)} bytes): {message}")
print(f"Cifrado PKCS1  ({len(ciphertext_pkcs1)} bytes): {ciphertext_pkcs1.hex()[:60]}...")

# Descifrar con clave privada
plaintext_pkcs1 = private_key.decrypt(
    ciphertext_pkcs1,
    asym_padding.PKCS1v15()
)

print(f"Descifrado     ({len(plaintext_pkcs1)} bytes): {plaintext_pkcs1}")
print(f"Correcto: {'✓' if plaintext_pkcs1 == message else '✗'}")

<a id='4-3'></a>
### 4.3 OAEP (Optimal Asymmetric Encryption Padding)

**OAEP** fue propuesto por Bellare y Rogaway (1994) y demostrado seguro en el modelo del oráculo aleatorio.

**Estructura del bloque OAEP** (tamaño bloque $= k$ bytes, hash $H$ de longitud $hLen$):

```
DB  = lHash || PS (ceros) || 0x01 || M          (k - hLen - 1 bytes)
seed = random(hLen bytes)
maskedDB   = DB   XOR MGF1(seed, k-hLen-1)
maskedSeed = seed XOR MGF1(maskedDB, hLen)
EM = 0x00 || maskedSeed || maskedDB
```

**MGF1** (*Mask Generation Function 1*) es una función de extensión de hash basada en SHA.

- OAEP es **IND-CCA2** seguro en el modelo de oráculo aleatorio.
- Longitud máxima del mensaje: $k - 2\cdot hLen - 2$ bytes.  
  Para RSA-2048 con SHA-256: $256 - 2 \times 32 - 2 = 190$ bytes.

In [ ]:
# Implementación educativa de MGF1 y OAEP (solo para entender, NO producción)

def mgf1(seed: bytes, length: int, hash_func=hashlib.sha256) -> bytes:
    """Mask Generation Function 1 (RFC 8017 B.2.1)."""
    hLen = hash_func(b'').digest_size
    if length > (2**32) * hLen:
        raise ValueError("mask too long")
    T = b''
    for counter in range(math.ceil(length / hLen)):
        C = counter.to_bytes(4, 'big')
        T += hash_func(seed + C).digest()
    return T[:length]

def oaep_encode(message: bytes, k: int, label: bytes = b'',
                hash_func=hashlib.sha256) -> bytes:
    """OAEP encode (RFC 8017 §7.1.1, sin la exponenciación RSA)."""
    hLen = hash_func(b'').digest_size
    mLen = len(message)
    if mLen > k - 2 * hLen - 2:
        raise ValueError(f"message too long: max {k-2*hLen-2} bytes")
    lHash = hash_func(label).digest()
    PS    = b'\x00' * (k - mLen - 2 * hLen - 2)
    DB    = lHash + PS + b'\x01' + message
    seed  = secrets.token_bytes(hLen)
    dbMask    = mgf1(seed, k - hLen - 1, hash_func)
    maskedDB  = bytes(a ^ b for a, b in zip(DB, dbMask))
    seedMask  = mgf1(maskedDB, hLen, hash_func)
    maskedSeed = bytes(a ^ b for a, b in zip(seed, seedMask))
    return b'\x00' + maskedSeed + maskedDB

def oaep_decode(em: bytes, k: int, label: bytes = b'',
                hash_func=hashlib.sha256) -> bytes:
    """OAEP decode (RFC 8017 §7.1.2, sin la exponenciación RSA)."""
    hLen = hash_func(b'').digest_size
    lHash = hash_func(label).digest()
    Y, maskedSeed, maskedDB = em[0], em[1:1+hLen], em[1+hLen:]
    assert Y == 0, "Y debe ser 0"
    seedMask = mgf1(maskedDB, hLen, hash_func)
    seed     = bytes(a ^ b for a, b in zip(maskedSeed, seedMask))
    dbMask   = mgf1(seed, k - hLen - 1, hash_func)
    DB       = bytes(a ^ b for a, b in zip(maskedDB, dbMask))
    # Verificar lHash
    assert DB[:hLen] == lHash, "hash de etiqueta incorrecto"
    # Encontrar 0x01 separador
    i = hLen
    while i < len(DB) and DB[i] == 0:
        i += 1
    assert DB[i] == 1, "separador 0x01 no encontrado"
    return DB[i+1:]

# Prueba educativa sobre el padding (k pequeño solo para demo)
k_demo = 64   # 512 bits — solo para la demo de padding
msg_demo = b"RSA-OAEP demo"
em = oaep_encode(msg_demo, k_demo)
decoded = oaep_decode(em, k_demo)

print(f"Mensaje original  : {msg_demo}")
print(f"EM codificado     : {em.hex()}")
print(f"Mensaje decodificado: {decoded}")
print(f"Correcto: {'✓' if decoded == msg_demo else '✗'}")

In [ ]:
# OAEP con cryptography — uso real
message_oaep = b"Datos sensibles cifrados con OAEP"

ct_oaep = public_key.encrypt(
    message_oaep,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

pt_oaep = private_key.decrypt(
    ct_oaep,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"Mensaje    ({len(message_oaep)} B): {message_oaep}")
print(f"Cifrado    ({len(ct_oaep)} B): {ct_oaep.hex()[:64]}...")
print(f"Descifrado ({len(pt_oaep)} B): {pt_oaep}")
print(f"Correcto: {'✓' if pt_oaep == message_oaep else '✗'}")

# Demostrar probabilismo: dos cifrados del mismo mensaje son distintos
ct2 = public_key.encrypt(message_oaep, asym_padding.OAEP(
    mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(), label=None))
print(f"\nDos cifrados del mismo mensaje son iguales: {ct_oaep == ct2}  → IND-CPA ✓")

In [ ]:
# Límites de tamaño para RSA-2048
import math

key_sizes = [1024, 2048, 3072, 4096]
hash_sizes = {'SHA-1': 20, 'SHA-256': 32, 'SHA-384': 48, 'SHA-512': 64}

print(f"{'Clave':>8}  {'Hash':>10}  {'Max OAEP (B)':>14}  {'Max PKCS1 (B)':>14}")
print("-" * 56)
for bits in key_sizes:
    k = bits // 8
    max_pkcs1 = k - 11
    for hname, hlen in hash_sizes.items():
        max_oaep = k - 2 * hlen - 2
        if max_oaep > 0:
            print(f"{bits:>8}  {hname:>10}  {max_oaep:>14}  {max_pkcs1:>14}")

---
<a id='5'></a>
## 5. Firma digital con RSA

En la firma digital RSA la clave privada **firma** y la clave pública **verifica**.  
El proceso siempre opera sobre el **hash** del mensaje, no sobre el mensaje completo.

$$\text{Firma: } \sigma = H(m)^d \bmod n$$
$$\text{Verificación: } H(m) \stackrel{?}{=} \sigma^e \bmod n$$

<a id='5-1'></a>
### 5.1 PKCS#1 v1.5 — firma

Añade el identificador del hash (OID) al bloque:

```
EM = 0x00 || 0x01 || PS (0xFF bytes) || 0x00 || DigestInfo
DigestInfo = ASN.1(OID_hash, Hash(M))
```

In [ ]:
# Firma PKCS#1 v1.5 con cryptography
document = b"Contrato de compraventa - Clausula 3: el vendedor se compromete..."

# Firmar
signature_pkcs1 = private_key.sign(
    document,
    asym_padding.PKCS1v15(),
    hashes.SHA256()
)

print(f"Documento ({len(document)} B): {document[:50]}...")
print(f"Firma PKCS1v15 ({len(signature_pkcs1)} B): {signature_pkcs1.hex()[:64]}...")

# Verificar
try:
    public_key.verify(
        signature_pkcs1,
        document,
        asym_padding.PKCS1v15(),
        hashes.SHA256()
    )
    print("Verificación: ✓ Firma válida")
except InvalidSignature:
    print("Verificación: ✗ Firma INVÁLIDA")

# Intentar verificar con documento alterado
try:
    public_key.verify(
        signature_pkcs1,
        document + b" (alterado)",
        asym_padding.PKCS1v15(),
        hashes.SHA256()
    )
    print("Verificación alterado: ✓ (ERROR — no debería pasar)")
except InvalidSignature:
    print("Verificación alterado: ✗ Detectada alteración ✓")

<a id='5-2'></a>
### 5.2 PSS (Probabilistic Signature Scheme)

**PSS** fue propuesto por Bellare y Rogaway (1996) y es la alternativa moderna recomendada.  
A diferencia de PKCS#1 v1.5, PSS tiene **prueba de seguridad** en el modelo del oráculo aleatorio.

**Estructura del encoding PSS:**

```
M'   = 0x00 * 8 || H(M) || salt        (salt aleatorio de sLen bytes)
H'   = H(M')
DB   = 0x00 * (emLen-sLen-hLen-2) || 0x01 || salt
dbMask    = MGF1(H', emLen-hLen-1)
maskedDB  = DB XOR dbMask
EM   = maskedDB || H' || 0xBC
```

- El **salt aleatorio** hace que cada firma sea distinta (probabilista).
- `sLen` recomendado: igual que `hLen` (p.ej. 32 bytes para SHA-256).

In [ ]:
# Firma PSS con cryptography
from cryptography.hazmat.primitives.asymmetric.padding import calculate_max_pss_salt_length

document = b"Contrato certificado con PSS"

# Firmar con PSS (salt = longitud del hash → máxima seguridad)
signature_pss = private_key.sign(
    document,
    asym_padding.PSS(
        mgf=asym_padding.MGF1(hashes.SHA256()),
        salt_length=asym_padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print(f"Firma PSS ({len(signature_pss)} B): {signature_pss.hex()[:64]}...")

# Demostrar que PSS es probabilista: dos firmas del mismo documento difieren
sig2 = private_key.sign(document,
    asym_padding.PSS(mgf=asym_padding.MGF1(hashes.SHA256()),
                     salt_length=asym_padding.PSS.MAX_LENGTH),
    hashes.SHA256())

print(f"Firmas distintas para mismo doc: {signature_pss != sig2}  → probabilismo ✓")

# Verificar
try:
    public_key.verify(
        signature_pss, document,
        asym_padding.PSS(mgf=asym_padding.MGF1(hashes.SHA256()),
                         salt_length=asym_padding.PSS.MAX_LENGTH),
        hashes.SHA256()
    )
    print("Verificación PSS: ✓ Firma válida")
except InvalidSignature:
    print("Verificación PSS: ✗ Firma INVÁLIDA")

<a id='5-3'></a>
### 5.3 Comparativa PKCS#1 v1.5 vs PSS

| Característica | PKCS#1 v1.5 | PSS |
|---|---|---|
| Determinista | ✓ Sí | ✗ No (salt aleatorio) |
| Prueba de seguridad formal | ✗ No | ✓ Sí (ROM) |
| Interoperabilidad | Muy alta (legado) | Alta (RFC 8017) |
| Recomendado para nuevos sistemas | ✗ No | ✓ Sí |
| Vulnerable a ataques de padding | Potencialmente | No |
| Usado en TLS 1.3 | ✗ (depreciado) | ✓ |

> **Regla de oro:** Usar siempre **PSS para firmas** y **OAEP para cifrado** en nuevos desarrollos.

---
<a id='6'></a>
## 6. RSA con la biblioteca `cryptography`

<a id='6-1'></a>
### 6.1 Generación, serialización y carga de claves

In [ ]:
# ── Generar clave RSA 2048 bits ─────────────────────────────────────────────
priv_2048 = generate_private_key(public_exponent=65537, key_size=2048)
pub_2048  = priv_2048.public_key()

# ── Serializar a PEM cifrado con contraseña ──────────────────────────────────
passphrase = b"mi_contrasena_segura_123"

pem_enc = priv_2048.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.BestAvailableEncryption(passphrase)
)
print("Clave privada cifrada (PEM):")
print(pem_enc.decode()[:200], "\n...")

# ── Cargar clave desde PEM ───────────────────────────────────────────────────
loaded_priv = serialization.load_pem_private_key(pem_enc, password=passphrase)
print("\nClave cargada correctamente:", loaded_priv.key_size, "bits ✓")

# ── Serializar clave pública a DER ───────────────────────────────────────────
der_pub = pub_2048.public_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)
print(f"Clave pública DER: {len(der_pub)} bytes, inicio: {der_pub[:8].hex()}")

<a id='6-2'></a>
### 6.2 Cifrado OAEP completo

In [ ]:
# Cifrado OAEP con SHA-256 y etiqueta personalizada
secret = b"Clave AES efimera: " + secrets.token_bytes(32)
label  = b"contexto=transferencia_bancaria"

ct = pub_2048.encrypt(
    secret,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=label
    )
)

pt = loaded_priv.decrypt(
    ct,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=label
    )
)

print(f"Secreto original ({len(secret)} B): {secret.hex()}")
print(f"Cifrado OAEP     ({len(ct)} B): {ct.hex()[:60]}...")
print(f"Recuperado       ({len(pt)} B): {pt.hex()}")
print(f"Correcto: {'✓' if pt == secret else '✗'}")

<a id='6-3'></a>
### 6.3 Firma y verificación PSS

In [ ]:
# Pipeline completo: firma → transmisión → verificación
import json as _json

# Simular un documento JSON a firmar
payload = _json.dumps({
    "emisor": "Banco XYZ",
    "receptor": "Cliente ABC",
    "monto": 15000.00,
    "moneda": "USD",
    "timestamp": "2026-04-05T00:00:00Z"
}).encode()

# Firmar
sig = priv_2048.sign(
    payload,
    asym_padding.PSS(
        mgf=asym_padding.MGF1(hashes.SHA256()),
        salt_length=asym_padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

# Empaquetar mensaje firmado
signed_msg = {
    "payload": base64.b64encode(payload).decode(),
    "signature": base64.b64encode(sig).decode(),
    "algorithm": "RSA-PSS-SHA256"
}
print("Mensaje firmado (JSON):")
print(_json.dumps(signed_msg, indent=2)[:300], "...")

# Verificar (lado receptor)
recv_payload = base64.b64decode(signed_msg["payload"])
recv_sig     = base64.b64decode(signed_msg["signature"])
try:
    pub_2048.verify(recv_sig, recv_payload,
        asym_padding.PSS(mgf=asym_padding.MGF1(hashes.SHA256()),
                         salt_length=asym_padding.PSS.MAX_LENGTH),
        hashes.SHA256())
    print("\nVerificación: ✓ Firma válida — mensaje íntegro y auténtico")
except InvalidSignature:
    print("\nVerificación: ✗ Firma INVÁLIDA")

<a id='6-4'></a>
### 6.4 Cifrado híbrido RSA + AES-GCM

Este es el patrón estándar para cifrar mensajes arbitrariamente largos con RSA.

In [ ]:
def hybrid_encrypt(public_key, plaintext: bytes) -> dict:
    """Cifrado híbrido RSA-OAEP + AES-256-GCM."""
    # 1. Generar clave de sesión AES-256
    aes_key = secrets.token_bytes(32)   # 256 bits
    # 2. Cifrar clave de sesión con RSA-OAEP
    enc_key = public_key.encrypt(
        aes_key,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    # 3. Cifrar datos con AES-256-GCM
    aesgcm = AESGCM(aes_key)
    nonce  = secrets.token_bytes(12)    # 96 bits
    ct_data = aesgcm.encrypt(nonce, plaintext, None)

    return {
        "enc_key": base64.b64encode(enc_key).decode(),
        "nonce":   base64.b64encode(nonce).decode(),
        "ct":      base64.b64encode(ct_data).decode(),
    }

def hybrid_decrypt(private_key, package: dict) -> bytes:
    """Descifrado híbrido RSA-OAEP + AES-256-GCM."""
    enc_key  = base64.b64decode(package["enc_key"])
    nonce    = base64.b64decode(package["nonce"])
    ct_data  = base64.b64decode(package["ct"])
    # 1. Recuperar clave AES con RSA
    aes_key = private_key.decrypt(
        enc_key,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    # 2. Descifrar datos con AES-GCM
    aesgcm = AESGCM(aes_key)
    return aesgcm.decrypt(nonce, ct_data, None)


# Prueba con un mensaje largo
large_msg = b"Este es un mensaje muy largo que excederia el limite de RSA solo. " * 100
print(f"Mensaje original: {len(large_msg)} bytes")

package = hybrid_encrypt(pub_2048, large_msg)
print(f"Clave cifrada RSA: {len(base64.b64decode(package['enc_key']))} bytes")
print(f"Datos cifrados AES-GCM: {len(base64.b64decode(package['ct']))} bytes")

recovered = hybrid_decrypt(priv_2048, package)
print(f"Mensaje recuperado: {len(recovered)} bytes")
print(f"Correcto: {'✓' if recovered == large_msg else '✗'}")

---
<a id='7'></a>
## 7. Ataques clásicos a RSA

Esta sección muestra **por qué** el uso correcto de RSA es tan importante. Cada ataque es consecuencia de una mala elección de parámetros o una implementación incorrecta.

<a id='7-1'></a>
### 7.1 Ataque de exponente público pequeño ($e = 3$, sin relleno)

Si $e = 3$ y el mensaje $m$ es pequeño respecto al módulo $n$, entonces:

$$c = m^3 \bmod n = m^3 \quad \text{(sin reducción modular)}$$

El atacante puede simplemente calcular $\sqrt[3]{c}$ sobre los enteros para recuperar $m$.

**Condición de ataque:** $m < n^{1/e}$

In [ ]:
def integer_cube_root(n: int) -> int:
    """Raíz cúbica entera de n usando búsqueda binaria."""
    if n < 0:
        return -integer_cube_root(-n)
    if n == 0:
        return 0
    # Estimación inicial
    x = int(round(n ** (1/3)))
    # Ajustar
    while (x+1)**3 <= n:
        x += 1
    while x**3 > n:
        x -= 1
    return x

def integer_eth_root(n: int, e: int) -> int:
    """Raíz e-ésima entera de n."""
    if n == 0:
        return 0
    x = int(round(n ** (1/e)))
    # refinar
    for _ in range(200):
        x1 = ((e-1)*x + n // pow(x, e-1)) // e
        if x1 >= x:
            break
        x = x1
    # verificar vecinos
    while pow(x+1, e) <= n:
        x += 1
    return x


# Escenario de ataque: e=3, mensaje pequeño
# Usamos un módulo RSA real pero con e=3 (¡nunca hacer esto en producción!)
e_small = 3
p_atk = generate_prime(512)
q_atk = generate_prime(512)
# Asegurar que e sea coprimo con lambda
while (p_atk - 1) % e_small == 0:
    p_atk = generate_prime(512)
while (q_atk - 1) % e_small == 0:
    q_atk = generate_prime(512)

n_atk = p_atk * q_atk

# Mensaje pequeño (< n^(1/3))
m_small = 12345678901234567890   # mucho menor que n^(1/3)
c_small = pow(m_small, e_small, n_atk)

# Ataque: la raíz cúbica de c es exactamente m (sin reducción modular)
m_recovered = integer_eth_root(c_small, e_small)

print(f"Módulo n ({n_atk.bit_length()} bits)")
print(f"n^(1/3) ≈ 10^{len(str(integer_eth_root(n_atk, 3)))}")
print(f"Mensaje m      : {m_small}  ({m_small.bit_length()} bits)")
print(f"¿m < n^(1/3)?  : {m_small < integer_eth_root(n_atk, 3)}")
print(f"Cifrado c      : {hex(c_small)[:30]}...")
print(f"m recuperado   : {m_recovered}")
print(f"¿c == m^3 (sin mod)? : {c_small == m_small**3}")
print(f"Ataque exitoso : {'✓' if m_recovered == m_small else '✗'}")
print("\n→ Con OAEP el relleno aleatorio hace m_padded >> n^(1/3), imposibilitando el ataque.")

<a id='7-2'></a>
### 7.2 Broadcast attack de Håstad

Si el mismo mensaje $m$ (sin relleno) se cifra para $e = 3$ destinatarios con módulos distintos $n_1, n_2, n_3$:

$$c_1 = m^3 \bmod n_1, \quad c_2 = m^3 \bmod n_2, \quad c_3 = m^3 \bmod n_3$$

Por el **Teorema Chino del Resto** (CRT), se puede reconstruir $m^3 \bmod (n_1 n_2 n_3)$, y luego aplicar la raíz cúbica.

In [ ]:
def crt(remainders, moduli):
    """Teorema Chino del Resto para una lista de (r_i, m_i)."""
    M  = 1
    for mi in moduli:
        M *= mi
    x  = 0
    for ri, mi in zip(remainders, moduli):
        Mi  = M // mi
        inv = pow(Mi, -1, mi)
        x  += ri * Mi * inv
    return x % M


# Tres receptores con módulos distintos, e=3, mismo mensaje sin relleno
e3 = 3
primes = []
for _ in range(6):
    pp = generate_prime(512)
    while pp % e3 == 0:
        pp = generate_prime(512)
    primes.append(pp)

n1 = primes[0] * primes[1]
n2 = primes[2] * primes[3]
n3 = primes[4] * primes[5]

m_bcast = 9876543210987654321   # mensaje pequeño
c1 = pow(m_bcast, e3, n1)
c2 = pow(m_bcast, e3, n2)
c3 = pow(m_bcast, e3, n3)

# Ataque CRT
m3_combined = crt([c1, c2, c3], [n1, n2, n3])
m_attack    = integer_eth_root(m3_combined, e3)

print(f"Mensaje original   : {m_bcast}")
print(f"m^3 mod (n1·n2·n3) recuperado correctamente: {m3_combined == m_bcast**3}")
print(f"Mensaje recuperado : {m_attack}")
print(f"Ataque exitoso     : {'✓' if m_attack == m_bcast else '✗'}")
print("\n→ OAEP usa relleno distinto para cada cifrado, imposibilitando este ataque.")

<a id='7-3'></a>
### 7.3 Factorización: Algoritmo $\rho$ de Pollard

Si $n = p \cdot q$ y uno de los factores es significativamente más pequeño que el otro (o tiene una estructura especial), el algoritmo $\rho$ de Pollard puede factorizar $n$ eficientemente.

La complejidad es $O(n^{1/4})$, que para un primo de 256 bits es ~$10^{19}$ — inviable para primos de 1024+ bits pero ilustrativo con ejemplos pequeños.

In [ ]:
def pollard_rho(n: int) -> int:
    """Algoritmo ρ de Pollard para factorizar n. Devuelve un factor no trivial."""
    if n % 2 == 0:
        return 2
    
    x = random.randint(2, n - 1)
    y = x
    c = random.randint(1, n - 1)
    d = 1
    
    while d == 1:
        x = (x * x + c) % n
        y = (y * y + c) % n
        y = (y * y + c) % n
        d = math.gcd(abs(x - y), n)
    
    return d if d != n else None   # None si falló (reintentar)

def factor_pollard(n: int, attempts: int = 100) -> tuple:
    """Factoriza n = p*q con múltiples intentos de Pollard ρ."""
    for _ in range(attempts):
        d = pollard_rho(n)
        if d and d != n:
            return d, n // d
    return None, None


# Ejemplo con módulo RSA de 64 bits (solo educativo)
p_small = generate_prime(32)
q_small = generate_prime(32)
n_small = p_small * q_small

print(f"n = {n_small}  ({n_small.bit_length()} bits)")
print(f"p real = {p_small}")
print(f"q real = {q_small}")

t0 = time.perf_counter()
p_found, q_found = factor_pollard(n_small)
elapsed = (time.perf_counter() - t0) * 1000

if p_found:
    print(f"\nPollard ρ encontró: p={p_found}, q={q_found}")
    print(f"Correcto: {'✓' if sorted([p_found,q_found])==sorted([p_small,q_small]) else '✗'}")
    print(f"Tiempo: {elapsed:.2f} ms")
else:
    print("Pollard ρ no convergió en este intento — intentar de nuevo")

print("\n→ Para n de 2048 bits, Pollard ρ tarda ~10^154 operaciones — impráctico.")
print("  El mejor algoritmo conocido (GNFS) tarda ~2^112 ops para 2048 bits.")

<a id='7-4'></a>
### 7.4 Ataque de módulo compartido (Common Modulus)

Si dos usuarios comparten el mismo módulo $n$ pero tienen exponentes distintos $e_1, e_2$ con $\gcd(e_1, e_2) = 1$, y el atacante tiene:

$$c_1 = m^{e_1} \bmod n, \quad c_2 = m^{e_2} \bmod n$$

Puede recuperar $m$ usando el algoritmo extendido de Euclides para encontrar $a, b$ tal que $a e_1 + b e_2 = 1$:

$$c_1^a \cdot c_2^b \equiv m^{ae_1 + be_2} = m^1 \equiv m \pmod{n}$$

In [ ]:
def extended_gcd(a: int, b: int) -> tuple:
    """gcd extendido: devuelve (g, x, y) con a*x + b*y = g."""
    if b == 0:
        return a, 1, 0
    g, x1, y1 = extended_gcd(b, a % b)
    return g, y1, x1 - (a // b) * y1

def mod_inverse_ext(a: int, m: int) -> int:
    g, x, _ = extended_gcd(a % m, m)
    assert g == 1, "no existe inverso"
    return x % m


# Generamos un módulo compartido
p_cm = generate_prime(256)
q_cm = generate_prime(256)
n_cm = p_cm * q_cm
phi_cm = (p_cm - 1) * (q_cm - 1)

# Dos pares de claves con mismo n
e1 = 17
e2 = 65537
assert math.gcd(e1, phi_cm) == 1
assert math.gcd(e2, phi_cm) == 1
assert math.gcd(e1, e2) == 1, "e1 y e2 deben ser coprimos para el ataque"

# Cifrar el mismo mensaje con ambas claves públicas
m_cm = int.from_bytes(b"secreto_comun", 'big')
c1_cm = pow(m_cm, e1, n_cm)
c2_cm = pow(m_cm, e2, n_cm)

# Ataque de módulo compartido
g, a, b = extended_gcd(e1, e2)
assert g == 1

# m = c1^a * c2^b mod n  (con manejo de exponentes negativos)
if a < 0:
    c1_inv = pow(c1_cm, -1, n_cm)
    part1 = pow(c1_inv, -a, n_cm)
else:
    part1 = pow(c1_cm, a, n_cm)

if b < 0:
    c2_inv = pow(c2_cm, -1, n_cm)
    part2 = pow(c2_inv, -b, n_cm)
else:
    part2 = pow(c2_cm, b, n_cm)

m_recovered = (part1 * part2) % n_cm

print(f"Mensaje original   : {m_cm}")
print(f"Mensaje recuperado : {m_recovered}")
print(f"Ataque exitoso     : {'✓' if m_recovered == m_cm else '✗'}")
print(f"\na*e1 + b*e2 = {a}*{e1} + ({b})*{e2} = {a*e1 + b*e2}")
print("\n→ Nunca compartir el módulo RSA entre dos entidades.")

<a id='7-5'></a>
### 7.5 Ataque de Wiener (exponente privado pequeño)

Wiener (1990) demostró que si $d < n^{0.25} / 3$, entonces $d$ puede recuperarse eficientemente a partir de la clave pública $(n, e)$ usando **fracciones continuas**.

La idea: $\frac{e}{n}$ tiene una fracción continua cuya lista de convergentes contiene $\frac{k}{d}$ (con error acotado), revelando $d$.

**Condición de vulnerabilidad:** $d < \frac{1}{3} n^{0.25}$

In [ ]:
def continued_fraction_convergents(n: int, d: int):
    """Genera convergentes de la fracción continua de n/d."""
    convergents = []
    while d:
        q = n // d
        convergents.append(q)
        n, d = d, n % d
    
    # Calcular convergentes h/k
    h0, h1 = 1, convergents[0]
    k0, k1 = 0, 1
    results = [(h1, k1)]
    for qi in convergents[1:]:
        h0, h1 = h1, qi * h1 + h0
        k0, k1 = k1, qi * k1 + k0
        results.append((h1, k1))
    return results

def wiener_attack(e: int, n: int):
    """Ataque de Wiener. Devuelve d si tiene éxito, None si no."""
    for k, d in continued_fraction_convergents(e, n):
        if k == 0:
            continue
        # Comprobar si phi = (e*d - 1) / k es un entero razonable
        if (e * d - 1) % k != 0:
            continue
        phi = (e * d - 1) // k
        # Verificar que n - phi + 1 tiene discriminante positivo (raíces reales)
        # n = p*q, phi = (p-1)(q-1) => p+q = n - phi + 1
        b = n - phi + 1
        disc = b * b - 4 * n
        if disc < 0:
            continue
        sqrt_disc = integer_eth_root(disc, 2)
        if sqrt_disc * sqrt_disc == disc:
            p = (b + sqrt_disc) // 2
            q = (b - sqrt_disc) // 2
            if p * q == n:
                return d, p, q
    return None, None, None


# Construir clave RSA con d pequeño (vulnerable a Wiener)
# Se elige d pequeño y se calcula e = d^-1 mod phi
p_w = generate_prime(256)
q_w = generate_prime(256)
n_w = p_w * q_w
phi_w = (p_w - 1) * (q_w - 1)

# d pequeño: d < n^0.25 / 3
threshold = integer_eth_root(n_w, 4) // 3
print(f"Umbral de Wiener: {threshold.bit_length()} bits")

# Generar d de ~60 bits (< umbral de 128 bits)
while True:
    d_w = random.getrandbits(60) | 1  # impar
    if math.gcd(d_w, phi_w) == 1:
        break

e_w = pow(d_w, -1, phi_w)
print(f"d pequeño ({d_w.bit_length()} bits): {d_w}")
print(f"e calculado ({e_w.bit_length()} bits): {hex(e_w)[:20]}...")
print(f"n ({n_w.bit_length()} bits): {hex(n_w)[:20]}...")

# Ataque
t0 = time.perf_counter()
d_found, p_found, q_found = wiener_attack(e_w, n_w)
elapsed = (time.perf_counter() - t0) * 1000

if d_found:
    print(f"\n✓ Wiener recuperó d = {d_found}")
    print(f"  Correcto: {d_found == d_w}")
    print(f"  p = {hex(p_found)[:20]}...")
    print(f"  Tiempo: {elapsed:.2f} ms")
else:
    print("✗ Wiener no tuvo éxito en este intento")
print("\n→ Usar siempre e=65537 y d de tamaño completo (≥ n^0.5).")

<a id='7-6'></a>
### 7.6 Ataque de Bleichenbacher (PKCS#1 v1.5 Padding Oracle)

El ataque de **Bleichenbacher** (1998) es el más devastador contra RSA-PKCS#1 v1.5 en la práctica.

**Premisa:** El servidor descifra con su clave privada y devuelve si el padding PKCS#1 v1.5 es válido (los primeros dos bytes son `0x00 0x02`). Esta pequeña fuga de información (**oráculo de padding**) permite un ataque adaptativo de texto cifrado elegido.

**Idea del ataque:**

1. El atacante tiene $c = m^e \bmod n$ (mensaje cifrado).
2. El atacante construye $c' = c \cdot s^e \bmod n$ para distintos $s$.
3. Consulta al oráculo si $c'$ descifra con padding válido.
4. Cada respuesta "sí/no" acota el espacio de búsqueda de $m$.
5. Tras ~$10^6$ consultas, se recupera $m$.

> Esta es la razón por la que **TLS 1.3** eliminó por completo RSA para el intercambio de claves. Versiones anteriores (TLS 1.0-1.2) eran vulnerables a variantes del ataque (DROWN, ROBOT).

**Mitigaciones:**
- Usar **OAEP** en lugar de PKCS#1 v1.5.
- Si se debe usar PKCS#1 v1.5, nunca revelar si el padding fue válido o no.
- Usar tiempo constante en la verificación del padding.

In [ ]:
# Demostración conceptual del oráculo de Bleichenbacher
# (no es el ataque completo — solo ilustra el concepto del oráculo)

def pkcs1v15_padding_oracle(private_key, ciphertext: bytes) -> bool:
    """Simula un oráculo de padding vulnerable: solo dice si el padding es válido.
    En un servidor real, esto podría inferirse por tiempo de respuesta o mensajes de error.
    """
    try:
        private_key.decrypt(ciphertext, asym_padding.PKCS1v15())
        return True   # padding correcto
    except Exception:
        return False  # padding incorrecto


# Cifrar un mensaje legítimo con PKCS#1 v1.5
msg_oracle = b"Secreto"
ct_oracle = pub_2048.encrypt(msg_oracle, asym_padding.PKCS1v15())

print("Demostración conceptual del oráculo de Bleichenbacher:")
print(f"Cifrado legítimo: {pkcs1v15_padding_oracle(priv_2048, ct_oracle)}")
print(f"Cifrado aleatorio: {pkcs1v15_padding_oracle(priv_2048, secrets.token_bytes(256))}")

# Mostrar maleabilidad RSA — multiplicar el cifrado por s^e
s = 2
n_pub = pub_2048.public_numbers().n
e_pub = pub_2048.public_numbers().e

# Construir cifrado maleable: c' = c * s^e mod n
ct_int    = int.from_bytes(ct_oracle, 'big')
s_e       = pow(s, e_pub, n_pub)
ct_mal_int = (ct_int * s_e) % n_pub
ct_maleable = ct_mal_int.to_bytes(256, 'big')

# El oráculo revela que el padding casi nunca es válido para un cifrado aleatorio
# pero el atacante repite miles de veces con distintos s hasta encontrar info útil
valid = pkcs1v15_padding_oracle(priv_2048, ct_maleable)
print(f"\nCifrado maleable con s={s}: padding {'VÁLIDO ← info útil' if valid else 'inválido'}")
print("\nEn el ataque real, se repite con miles de valores de s acotando m progresivamente.")
print("\n→ Usar siempre OAEP para cifrado RSA.")

---
<a id='8'></a>
## 8. Buenas prácticas y selección de parámetros

### 8.1 Tamaño de clave recomendado

| Año de retiro | Tamaño RSA | Seguridad equivalente |
|---|---|---|
| hasta 2023 | 1024 bits | 80 bits (**depreciado**) |
| hasta 2030 | 2048 bits | 112 bits |
| hasta 2040+ | 3072 bits | 128 bits |
| largo plazo | 4096 bits | 140 bits |
| post-cuántico | — | Kyber-768/1024, NTRU |

> Fuente: NIST SP 800-57 Part 1 Rev. 5

### 8.2 Checklist de implementación segura

| ✓ | Práctica |
|---|---|
| ✓ | Usar `e = 65537` como exponente público |
| ✓ | Usar primos `p`, `q` de tamaño igual (cada uno = mitad de n) |
| ✓ | Verificar que `p ≠ q` y que `gcd(e, λ(n)) = 1` |
| ✓ | Calcular `d` respecto a `λ(n)`, no a `φ(n)` |
| ✓ | Almacenar `p, q, dp, dq, qInv` para descifrado CRT |
| ✓ | **Cifrado:** usar OAEP con SHA-256 (no PKCS#1 v1.5 para nuevos sistemas) |
| ✓ | **Firma:** usar PSS con SHA-256 (no PKCS#1 v1.5 para nuevos sistemas) |
| ✓ | Usar tiempo constante al verificar padding y firmas |
| ✓ | Cifrar claves privadas en disco con passphrase fuerte |
| ✓ | Rotar claves periódicamente |
| ✗ | **NO** usar RSA sin relleno (textbook RSA) |
| ✗ | **NO** cifrar datos largos directamente con RSA — usar cifrado híbrido |
| ✗ | **NO** compartir módulos entre usuarios |
| ✗ | **NO** usar exponentes privados pequeños |

### 8.3 Alternativas modernas

Para nuevos sistemas, considerar:

- **ECDH + AES-GCM** para intercambio de clave y cifrado (menor overhead que RSA).
- **Ed25519 / ECDSA** para firma (más rápido y clave más corta que RSA-2048).
- **X25519** para Diffie-Hellman (más rápido y seguro que RSA).
- **Kyber / ML-KEM** para resistencia post-cuántica.

In [ ]:
# Comparativa de tiempos: RSA vs ECDH para generación e intercambio de clave
import timeit

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey

N = 20  # repeticiones

# RSA-2048 keygen
t_rsa_keygen = timeit.timeit(
    lambda: generate_private_key(65537, 2048),
    number=N
) / N * 1000

# RSA-2048 encrypt
t_rsa_enc = timeit.timeit(
    lambda: pub_2048.encrypt(b"A" * 32, asym_padding.OAEP(
        mgf=asym_padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(), label=None)),
    number=N
) / N * 1000

# RSA-2048 decrypt
ct_bench = pub_2048.encrypt(b"B" * 32, asym_padding.OAEP(
    mgf=asym_padding.MGF1(hashes.SHA256()),
    algorithm=hashes.SHA256(), label=None))
t_rsa_dec = timeit.timeit(
    lambda: priv_2048.decrypt(ct_bench, asym_padding.OAEP(
        mgf=asym_padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(), label=None)),
    number=N
) / N * 1000

# X25519 (ECDH moderno)
def x25519_exchange():
    a = X25519PrivateKey.generate()
    b = X25519PrivateKey.generate()
    a.exchange(b.public_key())

t_x25519 = timeit.timeit(x25519_exchange, number=N) / N * 1000

# ECDH P-256
def ecdh_p256():
    a = ec.generate_private_key(ec.SECP256R1())
    b = ec.generate_private_key(ec.SECP256R1())
    from cryptography.hazmat.primitives.asymmetric.ec import ECDH
    a.exchange(ECDH(), b.public_key())

t_ecdh = timeit.timeit(ecdh_p256, number=N) / N * 1000

print(f"{'Operación':<35} {'Tiempo (ms)':>12}")
print("-" * 50)
print(f"{'RSA-2048 keygen':<35} {t_rsa_keygen:>12.2f}")
print(f"{'RSA-2048 OAEP encrypt':<35} {t_rsa_enc:>12.2f}")
print(f"{'RSA-2048 OAEP decrypt':<35} {t_rsa_dec:>12.2f}")
print(f"{'X25519 full exchange':<35} {t_x25519:>12.2f}")
print(f"{'ECDH P-256 full exchange':<35} {t_ecdh:>12.2f}")
print(f"\nX25519 es {t_rsa_dec/t_x25519:.1f}× más rápido que RSA-2048 decrypt")

---
<a id='9'></a>
## 9. Referencias

### Estándares y RFCs

- **RFC 8017** — PKCS #1: RSA Cryptography Specifications Version 2.2  
  <https://www.rfc-editor.org/rfc/rfc8017>

- **FIPS 186-5** — Digital Signature Standard (DSS). NIST, 2023.  
  <https://csrc.nist.gov/publications/detail/fips/186/5/final>

- **NIST SP 800-57 Part 1 Rev. 5** — Recommendation for Key Management.  
  <https://csrc.nist.gov/publications/detail/sp/800-57-part-1/rev-5/final>

- **NIST SP 800-131A Rev. 2** — Transitioning the Use of Cryptographic Algorithms and Key Lengths.  
  <https://csrc.nist.gov/publications/detail/sp/800-131a/rev-2/final>

### Artículos originales

- **Rivest, R. L., Shamir, A., & Adleman, L.** (1978). A method for obtaining digital signatures and public-key cryptosystems. *Communications of the ACM*, 21(2), 120–126.

- **Bellare, M., & Rogaway, P.** (1994). Optimal asymmetric encryption. *EUROCRYPT 1994*, LNCS 950, 92–111.

- **Bellare, M., & Rogaway, P.** (1996). The exact security of digital signatures — How to sign with RSA and Rabin. *EUROCRYPT 1996*, LNCS 1070, 399–416.

- **Wiener, M. J.** (1990). Cryptanalysis of short RSA secret exponents. *IEEE Transactions on Information Theory*, 36(3), 553–558.

- **Bleichenbacher, D.** (1998). Chosen ciphertext attacks against protocols based on the RSA encryption standard PKCS #1. *CRYPTO 1998*, LNCS 1462, 1–12.

- **Håstad, J.** (1986). On using RSA with low exponent in a public key network. *CRYPTO 1985*, LNCS 218, 403–408.

- **Boneh, D., & Durfee, G.** (1999). Cryptanalysis of RSA with private key d less than N^0.292. *EUROCRYPT 1999*, LNCS 1592, 1–11.

### Libros de texto

- **Paar, C., & Pelzl, J.** (2009). *Understanding Cryptography*. Springer. Capítulos 7–8.

- **Boneh, D., & Shoup, V.** (2023). *A Graduate Course in Applied Cryptography*.  
  <https://toc.cryptobook.us/> (acceso libre)

- **Menezes, A., van Oorschot, P., & Vanstone, S.** (1996). *Handbook of Applied Cryptography*.  
  <https://cacr.uwaterloo.ca/hac/> (acceso libre)

- **Ferguson, N., Schneier, B., & Kohno, T.** (2010). *Cryptography Engineering*. Wiley.

### Herramientas y bibliotecas

- **Python `cryptography`**: <https://cryptography.io/en/latest/>
- **OpenSSL RSA**: <https://www.openssl.org/docs/man3.0/man1/openssl-rsa.html>
- **RsaCtfTool** (análisis de ataques CTF): <https://github.com/RsaCtfTool/RsaCtfTool>

---

> **Nota:** Esta clase es material educativo. Para implementaciones en producción, use siempre bibliotecas criptográficas auditadas como `cryptography`, `OpenSSL` o `libsodium`. Nunca implemente criptografía desde cero para datos reales.